In [ ]:
import numpy as np
import pandas as pd

from sktime.forecasting.structural import UnobservedComponents
from sktime.forecasting.ets import AutoETS
from sktime.forecasting.arima import StatsModelsARIMA as ARIMA


from sktime.utils.plotting import plot_series
# Modules for cross-validation
from sktime.split import temporal_train_test_split, ExpandingWindowSplitter, SlidingWindowSplitter, SingleWindowSplitter
from sktime.forecasting.model_evaluation import evaluate
from sktime.performance_metrics.forecasting import MeanSquaredError, MeanAbsoluteError, MeanAbsolutePercentageError # MSE, MAE, MAPE metrics

import pandas_datareader.data as web

# visualization settings
import matplotlib.pyplot as plt

# Do not show Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Do not show ValueWarning, ConvergenceWarning from statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

In [ ]:
z = web.DataReader(name='NA000334Q', data_source='fred', start='1990-01-01')
z.index = z.index.to_period(freq='Q')
y = np.log(z)

In [ ]:
# Define a list of specified forecasting models
forecasters = [ ARIMA(order=(0,1,0), seasonal_order=(1,0,1,4), trend='t'), 
				UnobservedComponents(level=True, trend=True, seasonal=4, cycle=False, 
				stochastic_level=True, stochastic_trend=True, stochastic_seasonal=True, stochastic_cycle=False), 
				AutoETS(error='add', trend='add', seasonal='add', damped_trend=False, auto=False, sp=4), 
				AutoETS(error='add', trend='add', seasonal='add', damped_trend=True, auto=False, sp=4) ]

# specify the cross-validation method. For example, SlidingWindowSplitter
cv_strategy = SlidingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=5)

# initialize metrics
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

# dataframe with metrics by columns
cv_data = pd.DataFrame(data=None, columns=['MSE', 'MAE', 'MAPE'])

for model in forecasters:
	print(model)
	cv_res = evaluate(forecaster=model, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
	# print(df.iloc[:,:len(metric)].mean()) # metrics for each model
	cv_data.loc[len(cv_data.index)] = cv_res.iloc[:,[0,1,2]].mean().values

# cross-validation results as a dataframe
cv_data

In [ ]:
# Model index
for i in range(cv_data.shape[1]):
	print(f'{cv_data.columns[i]}: model #={cv_data.iloc[:,i].argmin()}')